
# Unit 2 — Team Classification (Flights, BQML)

**Goal (team):** Build an *ops-ready* classifier in **BigQuery ML** to predict **`diverted`** on U.S. flights. Minimal handholding by design.

**What you deliver (inside this notebook):**
- One **LOGISTIC_REG** model (baseline), one **engineered** model using `TRANSFORM`
- **Evaluation** via `ML.EVALUATE` and **confusion matrices** (default 0.5 + your custom threshold)
- **Threshold choice** + 3–5 sentence ops justification
- Embedded **rubric** below (self-check before submission)

> Choose *one* dataset table that exists at your institution:  
> • `bigquery-public-data.faa.us_flights` **or** `bigquery-public-data.flights.*`  
> Make sure the table has `carrier`, `dep_delay`, `arr_delay` (for filters), `origin`, `dest`, `diverted` (or equivalent).


In [2]:
# --- Minimal setup (edit 3 vars) ---
from google.colab import auth
auth.authenticate_user()

import os
from google.cloud import bigquery

PROJECT_ID = "mgmt-467-471819"      # e.g., mgmt-467-47888
REGION     = "us-central1"
TABLE_PATH = "mgmt-467-471819.flights.flightdata"   # or your `bigquery-public-data.flights` table/view

os.environ["PROJECT_ID"] = PROJECT_ID
os.environ["REGION"]     = REGION
bq = bigquery.Client(project=PROJECT_ID)

print("BQ Project:", PROJECT_ID)
print("Source table:", TABLE_PATH)


BQ Project: mgmt-467-471819
Source table: mgmt-467-471819.flights.flightdata


### Quick sanity check

In [3]:

preview_sql = f"SELECT * FROM `{TABLE_PATH}` LIMIT 5"
bq.query(preview_sql).result().to_dataframe()


,Year,Quarter,Month,DayofMonth,DayOfWeek,FlightDate,Reporting_Airline,DOT_ID_Reporting_Airline,IATA_CODE_Reporting_Airline,Tail_Number,...,Div4WheelsOff,Div4TailNum,Div5Airport,Div5AirportID,Div5AirportSeqID,Div5WheelsOn,Div5TotalGTime,Div5LongestGTime,Div5WheelsOff,Div5TailNum
0,1998,3,8,29,6,1998-08-29,DL,19790,DL,N947DL,...,,,,None,None,,None,None,,
1,1996,3,7,14,7,1996-07-14,DL,19790,DL,N305DL,...,,,,None,None,,None,None,,
2,1994,1,2,19,6,1994-02-19,DL,19790,DL,,...,,,,None,None,,None,None,,
3,1995,3,7,16,7,1995-07-16,DL,19790,DL,N939DL,...,,,,None,None,,None,None,,
4,1994,3,7,5,2,1994-07-05,DL,19790,DL,,...,,,,None,None,,None,None,,



## 1) Canonical mapping (adjust as needed)
Map to a minimal schema used in the rest of the notebook:
- `flight_date` (DATE), `dep_delay` (NUM), `distance` (NUM), `carrier` (STRING), `origin` (STRING), `dest` (STRING), `diverted` (BOOL)


In [4]:
sql_data = f"""
CREATE OR REPLACE TABLE `{PROJECT_ID}.unit2_flights.split_data` AS
WITH canonical_flights AS (
  SELECT
    CAST(FlightDate AS DATE) AS flight_date,
    CAST(DepDelay AS FLOAT64) AS dep_delay,
    CAST(Distance AS FLOAT64) AS distance,
    CAST(Reporting_Airline AS STRING) AS carrier,
    CAST(Origin AS STRING) AS origin,
    CAST(Dest AS STRING) AS dest,
    CAST((CASE
        WHEN SAFE_CAST(Diverted AS INT64)=1 OR LOWER(CAST(Diverted AS STRING))='true'
        THEN TRUE ELSE FALSE END) AS BOOL) AS diverted
  FROM `{PROJECT_ID}.flights.flightdata`
  WHERE DepDelay IS NOT NULL
),
split AS (
  SELECT cf.*, CASE WHEN RAND()<0.8 THEN 'TRAIN' ELSE 'EVAL' END AS split
  FROM canonical_flights cf
)
SELECT * FROM split;
"""

bq.query(sql_data, job_config=bigquery.QueryJobConfig(use_legacy_sql=False)).result()
print("✅ Split table created")





✅ Split table created


### 2) Split (80/20)

In [5]:

SPLIT_CLAUSE = r'''
, split AS (
  SELECT cf.*,
         CASE WHEN RAND(12345) < 0.8 THEN 'TRAIN' ELSE 'EVAL' END AS split
  FROM canonical_flights cf
)
'''
print(SPLIT_CLAUSE)



, split AS (
  SELECT cf.*,
         CASE WHEN RAND(12345) < 0.8 THEN 'TRAIN' ELSE 'EVAL' END AS split
  FROM canonical_flights cf
)




## 3) Baseline model — LOGISTIC_REG (`diverted`)
Use **only** a small set of signals for the baseline (keep it honest).


In [6]:
MODEL_BASE = f"{PROJECT_ID}.unit2_flights.clf_diverted_base"
sql_model = f"""
CREATE OR REPLACE MODEL `{PROJECT_ID}.unit2_flights.clf_diverted_base`
OPTIONS(MODEL_TYPE='LOGISTIC_REG', INPUT_LABEL_COLS=['diverted']) AS
SELECT
  diverted,
  dep_delay,
  distance,
  carrier,
  origin,
  dest,
  EXTRACT(DAYOFWEEK FROM flight_date) AS day_of_week
FROM `{PROJECT_ID}.unit2_flights.split_data`
WHERE split='TRAIN';
"""

bq.query(sql_model, job_config=bigquery.QueryJobConfig(use_legacy_sql=False)).result()
print("✅ Model trained:", MODEL_BASE)



✅ Model trained: mgmt-467-471819.unit2_flights.clf_diverted_base


In [7]:
sql_evaluate = f"""
SELECT *
FROM ML.EVALUATE(
  MODEL `{PROJECT_ID}.unit2_flights.clf_diverted_base`,
  (
    SELECT
      diverted,
      dep_delay,
      distance,
      carrier,
      origin,
      dest,
      EXTRACT(DAYOFWEEK FROM flight_date) AS day_of_week
    FROM `{PROJECT_ID}.unit2_flights.split_data`
    WHERE split = 'EVAL'
  )
)
"""

job = bq.query(sql_evaluate, job_config=bigquery.QueryJobConfig(use_legacy_sql=False))
results = job.result().to_dataframe()
print("✅ Model evaluation complete:")
display(results)


✅ Model evaluation complete:


,precision,recall,accuracy,f1_score,log_loss,roc_auc
0,0.0,0.0,0.99756,0.0,0.016409,0.697421


### Confusion matrix — default 0.5 threshold

In [8]:

predict_sql = f"""
CREATE OR REPLACE TABLE `{PROJECT_ID}.unit2_flights.eval_predictions` AS
SELECT
  *
FROM ML.PREDICT(MODEL `{PROJECT_ID}.unit2_flights.clf_diverted_base`,
  (
    SELECT
      dep_delay,
      distance,
      carrier,
      origin,
      dest,
      EXTRACT(DAYOFWEEK FROM flight_date) AS day_of_week,
      diverted,
      split
    FROM `{PROJECT_ID}.unit2_flights.split_data`
    WHERE split = 'EVAL'
  )
)
"""

bq.query(predict_sql, job_config=bigquery.QueryJobConfig(use_legacy_sql=False)).result()
print("✅ Predictions saved to eval_predictions")




✅ Predictions saved to eval_predictions


In [9]:
cm_sql = f"""
SELECT
  SUM(CASE WHEN diverted = TRUE  AND predicted_diverted = TRUE  THEN 1 ELSE 0 END) AS TP,
  SUM(CASE WHEN diverted = FALSE AND predicted_diverted = TRUE  THEN 1 ELSE 0 END) AS FP,
  SUM(CASE WHEN diverted = TRUE  AND predicted_diverted = FALSE THEN 1 ELSE 0 END) AS FN,
  SUM(CASE WHEN diverted = FALSE AND predicted_diverted = FALSE THEN 1 ELSE 0 END) AS TN
FROM `{PROJECT_ID}.unit2_flights.eval_predictions`
"""

cm_df = bq.query(cm_sql).result().to_dataframe()
print("✅ Confusion matrix (efficient version):")
display(cm_df)


✅ Confusion matrix (efficient version):


,TP,FP,FN,TN
0,0,25,932,391185


### Confusion matrix — your custom threshold

In [10]:
CUSTOM_THRESHOLD = 0.2  # your chosen threshold

cm_thresh_sql = f"""
WITH scored AS (
  SELECT
    diverted AS label,
    CAST(predicted_diverted_probs[OFFSET(0)].prob >= {CUSTOM_THRESHOLD} AS BOOL) AS pred_label
  FROM `{PROJECT_ID}.unit2_flights.eval_predictions`
)
SELECT
  SUM(CASE WHEN label = TRUE  AND pred_label = TRUE  THEN 1 ELSE 0 END) AS TP,
  SUM(CASE WHEN label = FALSE AND pred_label = TRUE  THEN 1 ELSE 0 END) AS FP,
  SUM(CASE WHEN label = TRUE  AND pred_label = FALSE THEN 1 ELSE 0 END) AS FN,
  SUM(CASE WHEN label = FALSE AND pred_label = FALSE THEN 1 ELSE 0 END) AS TN
FROM scored
"""

cm_thresh_df = bq.query(cm_thresh_sql, job_config=bigquery.QueryJobConfig(use_legacy_sql=False)).result().to_dataframe()
print(f"✅ Confusion matrix at threshold = {CUSTOM_THRESHOLD}")
display(cm_thresh_df)



✅ Confusion matrix at threshold = 0.2


,TP,FP,FN,TN
0,1,40,931,391170



## 4) Engineered model — `TRANSFORM` (same label, stricter bar)
Create **route**, extract **day_of_week**, and **bucketize dep_delay**. Compare metrics to baseline.


In [11]:

MODEL_XFORM = f"{PROJECT_ID}.unit2_flights.clf_diverted_xform"

sql_xform = f"""
CREATE OR REPLACE MODEL `{MODEL_XFORM}`
TRANSFORM (
  -- Include label so BigQuery can access it
  diverted,

  -- Engineered features
  CONCAT(origin, '-', dest) AS route,
  EXTRACT(DAYOFWEEK FROM flight_date) AS day_of_week,
  CASE
    WHEN dep_delay <= -5 THEN 'early'
    WHEN dep_delay <= 5 THEN 'on_time'
    WHEN dep_delay <= 15 THEN 'minor'
    WHEN dep_delay <= 45 THEN 'moderate'
    ELSE 'major'
  END AS dep_delay_bucket,

  -- Keep raw numeric + categorical features
  dep_delay,
  distance,
  carrier,
  origin,
  dest
)
OPTIONS (
  MODEL_TYPE='LOGISTIC_REG',
  INPUT_LABEL_COLS=['diverted']
)
AS
SELECT *
FROM `{PROJECT_ID}.unit2_flights.split_data`
WHERE split = 'TRAIN';

-- Evaluate both models for comparison
SELECT 'baseline' AS model_version, *
FROM ML.EVALUATE(
  MODEL `{PROJECT_ID}.unit2_flights.clf_diverted_base`,
  (
    SELECT
      diverted,
      dep_delay,
      distance,
      carrier,
      origin,
      dest,
      EXTRACT(DAYOFWEEK FROM flight_date) AS day_of_week
    FROM `{PROJECT_ID}.unit2_flights.split_data`
    WHERE split = 'EVAL'
  )
)
UNION ALL
SELECT 'engineered' AS model_version, *
FROM ML.EVALUATE(
  MODEL `{MODEL_XFORM}`,
  (
    SELECT *
    FROM `{PROJECT_ID}.unit2_flights.split_data`
    WHERE split = 'EVAL'
  )
);
"""

job = bq.query(sql_xform, job_config=bigquery.QueryJobConfig(use_legacy_sql=False))
_ = job.result()
print("✅ Engineered model trained:", MODEL_XFORM)



✅ Engineered model trained: mgmt-467-471819.unit2_flights.clf_diverted_xform


In [12]:
MODEL_XFORM = f"{PROJECT_ID}.unit2_flights.clf_diverted_xform"

# Evaluate the engineered model on the EVAL split
eval_sql = f"""
SELECT
  *
FROM ML.EVALUATE(
  MODEL `{MODEL_XFORM}`,
  (
    SELECT *
    FROM `{PROJECT_ID}.unit2_flights.split_data`
    WHERE split = 'EVAL'
  )
);
"""

eval_xform_df = bq.query(eval_sql, job_config=bigquery.QueryJobConfig(use_legacy_sql=False)).result().to_dataframe()
print("✅ Evaluation metrics for engineered model:")
display(eval_xform_df)


✅ Evaluation metrics for engineered model:


,precision,recall,accuracy,f1_score,log_loss,roc_auc
0,0.0,0.0,0.997613,0.0,0.016247,0.679805


In [13]:
# Define model name
MODEL_XFORM = f"{PROJECT_ID}.unit2_flights.clf_diverted_xform"

# Run prediction once for evaluation split
pred_sql = f"""
CREATE OR REPLACE TABLE `{PROJECT_ID}.unit2_flights.eval_pred_xform` AS
SELECT
  *
FROM ML.PREDICT(
  MODEL `{MODEL_XFORM}`,
  (
    SELECT *
    FROM `{PROJECT_ID}.unit2_flights.split_data`
    WHERE split = 'EVAL'
  )
);
"""

bq.query(pred_sql, job_config=bigquery.QueryJobConfig(use_legacy_sql=False)).result()
print("✅ Predictions saved for engineered model.")

# Build confusion matrix
cm_sql = f"""
WITH scored AS (
  SELECT
    diverted AS label,
    predicted_diverted AS pred_label,
    predicted_diverted_probs[OFFSET(0)].prob AS score
  FROM `{PROJECT_ID}.unit2_flights.eval_pred_xform`
)
SELECT
  SUM(CASE WHEN label = TRUE  AND pred_label = TRUE  THEN 1 ELSE 0 END) AS TP,
  SUM(CASE WHEN label = FALSE AND pred_label = TRUE  THEN 1 ELSE 0 END) AS FP,
  SUM(CASE WHEN label = TRUE  AND pred_label = FALSE THEN 1 ELSE 0 END) AS FN,
  SUM(CASE WHEN label = FALSE AND pred_label = FALSE THEN 1 ELSE 0 END) AS TN
FROM scored;
"""

cm_xform_df = bq.query(cm_sql, job_config=bigquery.QueryJobConfig(use_legacy_sql=False)).result().to_dataframe()
print("✅ Confusion matrix for engineered model:")
display(cm_xform_df)


✅ Predictions saved for engineered model.
✅ Confusion matrix for engineered model:


,TP,FP,FN,TN
0,0,4,932,391206


In [14]:
MODEL_BASE = f"{PROJECT_ID}.unit2_flights.clf_diverted_base"
MODEL_XFORM = f"{PROJECT_ID}.unit2_flights.clf_diverted_xform"

compare_sql = f"""
-- Compare baseline and engineered model performance
SELECT
  'baseline' AS model_version,
  *
FROM ML.EVALUATE(
  MODEL `{MODEL_BASE}`,
  (
    SELECT
      diverted,
      dep_delay,
      distance,
      carrier,
      origin,
      dest,
      EXTRACT(DAYOFWEEK FROM flight_date) AS day_of_week
    FROM `{PROJECT_ID}.unit2_flights.split_data`
    WHERE split = 'EVAL'
  )
)
UNION ALL
SELECT
  'engineered' AS model_version,
  *
FROM ML.EVALUATE(
  MODEL `{MODEL_XFORM}`,
  (
    SELECT *
    FROM `{PROJECT_ID}.unit2_flights.split_data`
    WHERE split = 'EVAL'
  )
);
"""

compare_df = bq.query(compare_sql, job_config=bigquery.QueryJobConfig(use_legacy_sql=False)).result().to_dataframe()
print("✅ Baseline vs Engineered model comparison:")
display(compare_df)


✅ Baseline vs Engineered model comparison:


,model_version,precision,recall,accuracy,f1_score,log_loss,roc_auc
0,baseline,0.0,0.0,0.997560,0.0,0.016409,0.697420
1,engineered,0.0,0.0,0.997613,0.0,0.016247,0.679807



### Write-up (concise)
- **Threshold chosen & ops rationale:** …  
- **Baseline vs engineered — observed changes in AUC/precision/recall:** …  
- **Risk framing:** cost of FP vs FN for diversion planning; what is your acceptable FN-rate? …


The threshold I picked was 0.2. The main reason is that diversions are extremely rare in this dataset, so using the default 0.5 threshold basically makes the model predict “not diverted” every time. Missing a real diversion (a false negative) is worse than having a false positive, so lowering the threshold helps the model at least try to identify some of the rare diversion cases. It’s a simple way to make the model more useful without adding extra complexity.

For the baseline vs engineered comparison, the results honestly didn’t change that much. Accuracy stayed extremely high for both models, but that’s only because almost every flight is not diverted, so accuracy doesn’t really tell us much here. Precision and recall were both zero at the 0.5 threshold for both models, which again confirms the threshold issue. The ROC AUC dropped a little when using the engineered model (about 0.697 down to 0.680), meaning the engineered features didn’t actually improve how well the model ranks positive vs negative cases. Log loss got slightly better with the engineered version, so the probability estimates are a little more calibrated, but it’s not a huge difference.

The big thing to think about is the cost of false positives vs false negatives. A false negative means we miss a real diversion, which can mess up operations a lot more than accidentally flagging a flight that ends up being fine. A false positive just means a bit of overplanning. Because false negatives are more expensive, lowering the threshold makes sense, even if it increases false positives.

Overall, the engineered model didn’t drastically outperform the baseline. The main improvement actually comes from adjusting the threshold since the dataset is so imbalanced. Using something like 0.20 makes the model more sensitive, which is what you’d want in this kind of operational setting.


---

## Rubric (Flights, 100 pts)
**Team-only deliverable in this notebook**

- Baseline LOGISTIC_REG + evaluation (AUC + confusion @0.5) — **20**  
- Custom threshold confusion matrix + ops justification — **20**  
- Engineered model with `TRANSFORM` (route, DOW, delay bucket) — **20**  
- Comparison table (baseline vs engineered) + 3–5 sentence interpretation — **20**  
- Reproducibility: parameters clear, no hidden magic; schema mapping documented — **10**  
- Governance notes: assumptions/limitations + slices you would monitor — **10**

> **Strictness:** No screenshots; use actual results cells. Keep explanations concise (bullet points OK).
